# Train a ball detector on your own games

This notebook teaches a small model to find the basketball, using the frames you labelled on the
Hoops Tracker dashboard ("Train the model"). It runs on Google Colab's free graphics card, because
the laptop has none.

**Before you start**

1. On the dashboard, tab **Export**, press "Make the zip for Google Colab".
2. In Google Drive, make a folder called `hoops-tracker` and upload the zip (`ball_dataset_<number>.zip`) into it.
3. Here in Colab: menu **Runtime**, **Change runtime type**, choose **T4 GPU**, Save.
4. Menu **Runtime**, **Run all**. Colab asks for permission to use your Drive: allow it.

Leave this tab open until the last cell has printed its scores.

**Privacy.** The zip holds frames of the game, and they show the players. Uploading it to Drive
means those frames are kept by Google. This notebook reads them from your Drive and writes only the
finished model and a small text file back; it sends nothing anywhere else. Clear the outputs
(menu **Edit**, **Clear all outputs**) before you save or share a copy of this notebook: an output
can contain pictures from the game.

## 1. Settings

You should not need to change these.

- **The model, `yolo11n.pt`.** It is the smallest ("nano") official YOLO model: the laptop that will
  use it has no graphics card, so the model has to be small to get through a game in reasonable time.
- **The picture size, 1280.** The ball is only about 30 pixels wide in a 1920-pixel frame; at the usual
  size of 640 it would shrink to about 10 pixels and be lost, at 1280 it stays about 20 pixels wide.
  (The price: on the laptop each frame takes about four times as long as it would at 640.)

If the scores at the end are poor although you have labelled several hundred frames, the next thing to
try is the next size up, `yolo11s.pt`, which is more accurate and roughly three times slower.

In [ ]:
DRIVE_FOLDER = "/content/drive/MyDrive/hoops-tracker"   # the Drive folder you uploaded the zip to
WORK_FOLDER = "/content/ball_training"                  # working space on Colab; it is deleted when Colab closes

MODEL = "yolo11n.pt"     # official starting weights; they are downloaded by themselves
PICTURE_SIZE = 1280      # pictures are shrunk to this width for training and for testing
EPOCHS = 100             # how many times training goes through all the pictures (it stops early if nothing improves)
BATCH = 8                # pictures looked at together; halve it if Colab says it is out of memory

## 2. Connect Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Unpack the newest dataset

Takes the `ball_dataset_<number>.zip` with the highest number in the Drive folder. That number becomes
the version of the model: dataset 3 gives `ball_model_v3.pt`.

In [ ]:
import re
import zipfile
from pathlib import Path


def zip_number(path):
    return int(re.findall(r"\d+", path.stem)[-1])


zips = sorted(Path(DRIVE_FOLDER).glob("ball_dataset_*.zip"), key=zip_number)
assert zips, f"No ball_dataset_<number>.zip found in {DRIVE_FOLDER}. Upload the zip from the dashboard's Export tab there."
dataset_zip = zips[-1]
VERSION = zip_number(dataset_zip)

dataset = Path(WORK_FOLDER) / dataset_zip.stem
with zipfile.ZipFile(dataset_zip) as archive:
    archive.extractall(dataset)
DATA = str(dataset / "data.yaml")

print(f"Using {dataset_zip.name}")
print(Path(DATA).read_text())
for part in ("train", "val", "test"):
    pictures = len(list((dataset / "images" / part).glob("*.jpg")))
    with_ball = sum(1 for label in (dataset / "labels" / part).glob("*.txt") if label.read_text().strip())
    print(f"{part}: {pictures} pictures, {with_ball} with the ball marked")

## 4. Install the training program

Ultralytics is the program that trains YOLO models. The version is the one installed on the laptop, so
the model file is sure to open there.

In [ ]:
%pip install -q ultralytics==8.4.172

## 5. Train

Training shows the model the `train` pictures again and again, and after each round checks itself on
the `val` pictures. The `test` pictures are not touched here. With a few hundred pictures expect
some tens of minutes on the T4 GPU (this has not been timed yet). The best version found is kept as `best.pt`.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(data=DATA, imgsz=PICTURE_SIZE, epochs=EPOCHS, batch=BATCH, patience=30,
            project=f"{WORK_FOLDER}/runs", name=f"ball_v{VERSION}", exist_ok=True)
best = Path(model.trainer.best)
print(f"Best model: {best}")

## 6. The honest test

Now the best model is tried on the `test` pictures, which it has never seen.

- **Precision**: of the boxes the model drew, the share that really were the ball. Low precision means
  false alarms (elbows, heads, shoes taken for the ball).
- **Recall**: of the balls you marked, the share the model found. Low recall means missed balls.

If the test pictures are a whole game that was kept aside, these numbers are an honest measure. If they
are stretches of the same games the model trained on (the dataset says so at the top of `data.yaml`,
printed in step 3), they will look better than the model really is on a new game.

In [ ]:
best_model = YOLO(best)
test = best_model.val(data=DATA, split="test", imgsz=PICTURE_SIZE,
                      project=f"{WORK_FOLDER}/runs", name=f"ball_v{VERSION}_test", exist_ok=True)
precision, recall, map50 = float(test.box.mp), float(test.box.mr), float(test.box.map50)
print(f"Precision {precision:.2f}   Recall {recall:.2f}   mAP50 {map50:.2f}")

## 7. Save the model to Drive

Copies the model to the Drive folder as `ball_model_v<number>.pt`, with a small text file of its scores
next to it.

In [ ]:
import shutil
from datetime import date

import ultralytics

model_file = Path(DRIVE_FOLDER) / f"ball_model_v{VERSION}.pt"
shutil.copyfile(best, model_file)

notes = [line for line in Path(DATA).read_text().splitlines() if line.startswith("#")]
scores = "\n".join([
    f"ball_model_v{VERSION}.pt   trained {date.today().isoformat()} from {dataset_zip.name}",
    f"Started from {MODEL}, picture size {PICTURE_SIZE}, up to {EPOCHS} epochs, Ultralytics {ultralytics.__version__}",
    "",
    "Scores on the test pictures (never used in training):",
    f"  Precision {precision:.3f}   (of the boxes it drew, the share that were the ball)",
    f"  Recall    {recall:.3f}   (of the balls marked, the share it found)",
    f"  mAP50     {map50:.3f}   (one overall score from 0 to 1: higher is better)",
    "",
    "What the dataset said about itself:",
    *notes,
]) + "\n"
scores_file = Path(DRIVE_FOLDER) / f"ball_model_v{VERSION}_scores.txt"
scores_file.write_text(scores)

print(scores)
print(f"Saved {model_file}")
print(f"Saved {scores_file}")

## What next

1. In Google Drive, download `ball_model_v<number>.pt` (and the scores file) from the `hoops-tracker` folder.
2. Put the `.pt` file in the `models` folder on the SD card. It then shows up on the dashboard's **Models** tab.
3. Using the ball model when a game is processed is the next stage of the project. Until that is built, the
   model changes nothing in the stats.

More labelled frames, from more games and gyms, is the surest way to a better model: label more, export
again (the zip gets the next number) and run this notebook again.